# Overview Materi

Jelaskan perbedaan singkat antara grid, randomized, bayesian search cv dengan optuna menurut pemahamanmu

source: https://www.youtube.com/watch?v=t-INgABWULw

Grid mencoba semuanya, sangat lambat.

Randomized mencoba secara acak, lebih cepat tapi gambling.

Bayesian Search  CV mencoba secara cerdas dengan belajar dari pengalaman, sangat efisien dibanding metode yang lainnya.

# Import Data & Libraries

In [1]:
# jalankan hanya sekali
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 12.1 MB/s eta 0:00:00


In [2]:
# import library yang dibutuhkan di sini
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = sns.load_dataset('iris')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


# Data Preprocessing

In [4]:
# ubah variabel kategorik ke numerik
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['species'] = le.fit_transform(df['species'])

In [5]:
display(df.head())

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [33]:
# subsetting peubah
X = df.drop(['species'], axis=1)
y = df['species']


# Dataset Splitting

In [28]:
# split dengan rasio 80:20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


# Base Model Random Forest

In [23]:
# gunakan random forest classifier
rfr = RandomForestClassifier(random_state=42)
rfr.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = rfr.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.3f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.3f}")
print(classification_report(y_test, y_pred))

Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



# Optuna

In [27]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    max_depth = trial.suggest_int('max_depth', 2, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 32)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, n_jobs=-1, cv=5, scoring='accuracy')

    return -score.mean()

Hyperparameter dapat disesuaikan dengan algoritma yang digunakan. Kali ini kita menggunakan Random Forest sehingga yang dapat kita select adalah *n_estimators, max_depth, min_samples_split,* dan *min_samples_leaf*

In [31]:
study = optuna.create_study(direction='minimize')

[I 2025-10-04 16:22:47,362] A new study created in memory with name: no-name-7751e205-23a0-494e-b56e-aeef0633790c


In [32]:
study.optimize(objective, n_trials=100)

[I 2025-10-04 16:23:02,070] Trial 0 finished with value: -0.8583333333333332 and parameters: {'n_estimators': 219, 'max_depth': 44, 'min_samples_split': 29, 'min_samples_leaf': 23}. Best is trial 0 with value: -0.8583333333333332.
[I 2025-10-04 16:23:03,042] Trial 1 finished with value: -0.9416666666666667 and parameters: {'n_estimators': 120, 'max_depth': 25, 'min_samples_split': 3, 'min_samples_leaf': 10}. Best is trial 1 with value: -0.9416666666666667.
[I 2025-10-04 16:23:04,828] Trial 2 finished with value: -0.7583333333333333 and parameters: {'n_estimators': 262, 'max_depth': 24, 'min_samples_split': 24, 'min_samples_leaf': 26}. Best is trial 1 with value: -0.9416666666666667.
[I 2025-10-04 16:23:10,162] Trial 3 finished with value: -0.9083333333333332 and parameters: {'n_estimators': 781, 'max_depth': 18, 'min_samples_split': 25, 'min_samples_leaf': 22}. Best is trial 1 with value: -0.9416666666666667.
[I 2025-10-04 16:23:15,399] Trial 4 finished with value: -0.9416666666666667 

it may take a while... so just wait n see ^^
<br>
they recommend to set n_trials at 100 cz it seems there's no significant score increase after 100 trials (also inefficient too, you'll have to wait in a quite long time)

In [40]:
study.best_params

{'n_estimators': 636,
 'max_depth': 30,
 'min_samples_split': 2,
 'min_samples_leaf': 2}

Berikut hasil hyperparameter tuning dari Optuna

In [ ]:
# cek hasil hyperparameter tuning dari Optuna

Best Hyperparameters: {'n_estimators': 523, 'max_depth': 17, 'min_samples_split': 12, 'min_samples_leaf': 15}


In [43]:
print(f"Best Hyperparameters: {study.best_params}")

Best Hyperparameters: {'n_estimators': 636, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 2}


# Random Forest Using Optuna

In [45]:
# simpan hasil best hyperparameter tuning ke variabel bari
best_params = study.best_params

In [46]:
best_model = RandomForestClassifier(**best_params, random_state=42)

best_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=30, min_samples_leaf=2, n_estimators=636,
                       random_state=42)

In [47]:
y_pred = best_model.predict(X_test)

In [48]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.3f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted'):.3f}")
print(classification_report(y_test, y_pred))

Accuracy: 1.000
Precision: 1.000
Recall: 1.000
F1 Score: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



Tidak terdapat kenaikan skor dengan sebelum menggunakan Optuna sebab skor yang dihasilkan melalui base model saja sudah bagus